In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [22]:
# Load datasets 
fear_df = pd.read_csv("fear_greed_index.csv")
trader_df = pd.read_csv("historical_data.csv")

# parse dates
fear_df['date'] = pd.to_datetime(fear_df['date'], errors='coerce')
trader_df['Timestamp IST'] = pd.to_datetime(trader_df['Timestamp IST'], errors='coerce')
trader_df['Timestamp'] = pd.to_datetime(trader_df['Timestamp'], errors='coerce')

In [ ]:
print(trader_df['Timestamp IST'].head(10))

0   2024-02-12 22:50:00
1   2024-02-12 22:50:00
2   2024-02-12 22:50:00
3   2024-02-12 22:50:00
4   2024-02-12 22:50:00
5   2024-02-12 22:50:00
6   2024-02-12 22:50:00
7   2024-02-12 22:50:00
8   2024-02-12 22:50:00
9   2024-02-12 22:50:00
Name: Timestamp IST, dtype: datetime64[ns]


Data Cleaning

In [ ]:
print("Details of fear_greed_index dataset")
print(f"Shape (rows, columns): {fear_df.shape}")
print(f"Columns: {list(fear_df.columns)}")

print("Details of historical_data dataset")
print(f"Shape (rows, columns): {trader_df.shape}")
print(f"Columns: {list(trader_df.columns)}")


Details of fear_greed_index dataset
Shape (rows, columns): (2644, 4)
Columns: ['timestamp', 'value', 'classification', 'date']
Details of historical_data dataset
Shape (rows, columns): (211224, 16)
Columns: ['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side', 'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL', 'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID', 'Timestamp']


In [13]:

# fear_df.isnull().sum()
trader_df.isnull().sum()


Account                  0
Coin                     0
Execution Price          0
Size Tokens              0
Size USD                 0
Side                     0
Timestamp IST       131999
Start Position           0
Direction                0
Closed PnL               0
Transaction Hash         0
Order ID                 0
Crossed                  0
Fee                      0
Trade ID                 0
Timestamp                0
dtype: int64

In [27]:
print("Duplicated in fear_gread_index : ",trader_df.duplicated().sum())
print("Duplicated in historical_data : ",fear_df.duplicated().sum())

Duplicated in fear_gread_index :  0
Duplicated in historical_data :  0


2) Convert timestamps and align the datasets by date (daily level is fine).

In [ ]:
fear_df['date'] = pd.to_datetime(fear_df['date'], errors='coerce')
trader_df['Timestamp IST'] = (
    trader_df['Timestamp IST']
    .astype(str)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

trader_df['Timestamp IST'] = pd.to_datetime(
    trader_df['Timestamp IST'],
    dayfirst=True,
    errors='coerce'
)

trader_df['date_only'] = trader_df['Timestamp IST'].dt.date
daily_trader = trader_df.groupby('date_only').agg({
    'Closed PnL': 'sum',
    'Size USD': 'sum',
    'Trade ID': 'count'
}).reset_index()




# daily_sentiment = fear_df[['date_only', 'value', 'classification']].copy()
# merged_df = pd.merge(
#     daily_trader,
#     daily_sentiment,
#     on='date_only',
#     how='inner'   # use inner to keep only matching dates
# )
# print(merged_df.head())
# print("Rows after merge:", len(merged_df))

Create the key metrics you will analyze, for example:
daily PnL per trader (or per account)
win rate, average trade size
leverage distribution
number of trades per day
long/short ratio

In [30]:
numeric_cols = ['Execution Price', 'Size Tokens', 'Size USD', 'Closed PnL', 'Fee']

for col in numeric_cols:
    trader_df[col] = pd.to_numeric(trader_df[col], errors='coerce')

In [31]:
# Daily PnL per account
daily_pnl_account = trader_df.groupby(['date_only', 'Account'])['Closed PnL'].sum().reset_index()

daily_pnl_account.rename(columns={'Closed PnL': 'daily_pnl'}, inplace=True)

In [32]:
# Win rate
trader_df['is_win'] = trader_df['Closed PnL'] > 0

win_rate_account = trader_df.groupby('Account')['is_win'].mean().reset_index()
win_rate_account.rename(columns={'is_win': 'win_rate'}, inplace=True)

In [33]:
# Number of trades per day
trades_per_day = trader_df.groupby('date_only')['Trade ID'].count().reset_index()

trades_per_day.rename(columns={'Trade ID': 'trade_count'}, inplace=True)

In [34]:
# Long vs Short ratio
trader_df['Direction'] = trader_df['Direction'].str.lower().str.strip()

long_short = trader_df.groupby(['date_only', 'Direction'])['Trade ID'].count().unstack().fillna(0)

long_short['long_short_ratio'] = long_short.get('long', 0) / (long_short.get('short', 1))

long_short = long_short.reset_index()

In [ ]:
# Leverage (approximation)
trader_df['leverage_est'] = trader_df['Size USD'] / trader_df['Execution Price']

leverage_dist = trader_df['leverage_est'].describe()
print(leverage_dist)

count    2.112240e+05
mean     4.623368e+03
std      1.042730e+05
min      0.000000e+00
25%      2.939983e+00
50%      3.199997e+01
75%      1.879014e+02
max      1.582244e+07
Name: leverage_est, dtype: float64


Part B : Analysis (must-have)

In [37]:
fear_df['date_only'] = fear_df['date'].dt.date
merged_df = pd.merge(
    trader_df,
    fear_df[['date_only', 'classification']],
    on='date_only',
    how='inner'
)
filtered_df = merged_df[
    merged_df['classification'].isin(['Fear', 'Greed'])
].copy()
filtered_df['is_win'] = filtered_df['Closed PnL'] > 0
filtered_df['loss'] = filtered_df['Closed PnL'].apply(lambda x: x if x < 0 else 0)
performance_by_sentiment = filtered_df.groupby('classification').agg({
    'Closed PnL': ['mean', 'sum'],
    'is_win': 'mean',
    'loss': 'mean',
    'Trade ID': 'count'
}).reset_index()

performance_by_sentiment.columns = [
    'sentiment',
    'avg_pnl',
    'total_pnl',
    'win_rate',
    'avg_loss',
    'trade_count'
]

print(performance_by_sentiment)
performance_by_sentiment['pnl_per_trade'] = (
    performance_by_sentiment['total_pnl'] /
    performance_by_sentiment['trade_count']
)
filtered_df.groupby('classification')['Closed PnL'].describe()
filtered_df.groupby('classification')['Closed PnL'].quantile([0.1, 0.25, 0.5])

  sentiment     avg_pnl     total_pnl  win_rate   avg_loss  trade_count
0      Fear  128.287950  1.779226e+06  0.381787 -14.324172        13869
1     Greed   53.988003  6.096325e+05  0.435707 -32.370975        11292


classification      
Fear            0.10    0.0
                0.25    0.0
                0.50    0.0
Greed           0.10    0.0
                0.25    0.0
                0.50    0.0
Name: Closed PnL, dtype: float64